# Portfolio Management for Beginners: Optimization & Factor Analysis

**Instructions:** Read through this notebook. It is designed for absolute beginners who have never heard of portfolio management. We will explore how to build a smart collection of stocks (a portfolio) and how to understand the hidden risks inside it.

## Section 0: Gathering Our Stocks
### 📝 What is a Portfolio?
A **portfolio** is simply a collection of investments. Instead of putting all your money into one stock (like Apple), you spread it around into a "basket" of many different stocks. 

Before we can figure out the best way to mix our stocks, we need to pick which stocks we want in our basket. Here, we choose 25 well-known companies from 5 different areas of the economy (Technology, Finance, Healthcare, Energy, and Industrials).

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Our basket of 25 stocks, grouped by their industry
sector_map = {
    'Tech': ['AAPL', 'MSFT', 'GOOGL', 'NVDA', 'AMD'],
    'Financials': ['JPM', 'BAC', 'WFC', 'GS', 'MS'],
    'Healthcare': ['JNJ', 'PFE', 'UNH', 'ABBV', 'MRK'],
    'Energy': ['XOM', 'CVX', 'COP', 'SLB', 'EOG'],
    'Industrials': ['CAT', 'DE', 'GE', 'HON', 'LMT']
}
tickers = [ticker for group in sector_map.values() for ticker in group]


Now, we will download the daily price history for these 25 stocks over a three-year period (2021 to 2024).

In [ ]:
print("Downloading price history for our 25 stocks...")
prices = yf.download(tickers, start="2021-01-01", end="2024-01-01")['Close']
prices = prices.dropna()
print("Done downloading!")


### 📝 Measuring Returns and Relationships
Now that we have the prices, we need to calculate two very important things:
1.  **Returns**: A return is simply the percentage of money a stock made or lost each day. 
2.  **The Covariance Matrix**: This is like a "friendship chart" for stocks. It measures how the stocks move together. If Apple and Microsoft usually go up on the same days, they have a positive relationship. If one goes up while the other goes down, they have a negative relationship. This "friendship chart" is the secret to building a great portfolio.

In [ ]:
# Calculate daily percentage returns
returns = np.log(prices / prices.shift(1)).dropna()

# Calculate the average yearly returns and the "friendship chart" (covariance matrix)
mean_returns = returns.mean() * 252
cov_matrix = returns.cov() * 252


## Section 1: Finding the Perfect Mix (Optimization)
### 📝 Why not just split our money equally?
If you have $10,000, you could just put an equal amount into all 25 stocks. But is that the smartest way? 

What if 10 of those stocks always crash at the exact same time? If you own all 10, your whole portfolio will crash. Instead, you want to mix stocks that move differently. When some go down, others might go up, which smooths out your ride. This is called **diversification**.

**Mean-Variance Optimization (MVO)** is a mathematical recipe for finding the absolute best mix of stocks. It looks at the "friendship chart" (covariance matrix) and tries to find the combination of stocks that gives you the highest possible reward for the lowest possible bumpiness (risk). 

Below, we ask the computer to solve this puzzle and find the perfect percentage (weight) to invest in each stock.

In [ ]:
# We use a mathematical shortcut to find the best possible mix of stocks
inv_cov = np.linalg.inv(cov_matrix)
excess_returns = mean_returns - 0.02 # We assume we could get a safe 2% return from a bank

# The computer calculates the raw mix
raw_weights = np.dot(inv_cov, excess_returns)

# We adjust the numbers so they equal exactly 100% of our money
optimal_weights = raw_weights / np.sum(raw_weights)


Now let's see how good our "perfect mix" is. We will look at its expected return, its risk (how bumpy the ride will be), and its Sharpe Ratio (a score that tells us how much reward we are getting for every ounce of risk).

In [ ]:
portfolio_return = np.sum(optimal_weights * mean_returns)
portfolio_vol = np.sqrt(np.dot(optimal_weights.T, np.dot(cov_matrix, optimal_weights)))
sharpe_ratio = (portfolio_return - 0.02) / portfolio_vol

print(f"Expected Yearly Reward: {portfolio_return:.2%}")
print(f"Expected Bumpiness (Risk): {portfolio_vol:.2%}")
print(f"Reward-to-Risk Score (Sharpe Ratio): {sharpe_ratio:.2f}")


Let's visualize the recipe. This chart shows exactly what percentage of our money should go into each of the 25 stocks to achieve that perfect balance.

In [ ]:
# Plot the exact percentage to invest in each stock
plt.figure(figsize=(12, 5))
pd.Series(optimal_weights, index=tickers).plot(kind='bar', color='skyblue', edgecolor='black')
plt.title("The Perfect Mix: How much to invest in each stock")
plt.ylabel("Percentage of our Total Money")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


## Section 2: Understanding Hidden Risks (Factor Analysis)
### 📝 What is a Hidden Factor?
Imagine you are watching a bunch of leaves blowing around in the air. Some blow left, some blow right, some blow up. But there is a hidden, underlying force driving most of their movement: the wind. 

In the stock market, there are "winds" that push stocks around. We call these hidden forces **Factors**. Instead of trying to understand the complex story of 25 different companies, we can use a tool called **Principal Component Analysis (PCA)** to find the main "winds" moving the whole market.

*   **The Strongest Wind (Factor 1):** Usually, this just represents the overall mood of the stock market. If the whole market goes up, most stocks go up.
*   **The Second Wind (Factor 2):** This might represent something more specific, like technology companies having a good day while oil companies have a bad day.

### 📝 What is a Factor Loading?
A **Factor Loading** simply measures how sensitive a specific stock is to a specific "wind". 
*   If Apple has a high loading on "Wind 1", it means Apple gets pushed very hard when that wind blows.
The PCA tool gives us a table (a "Loadings Matrix") that shows every stock's sensitivity to every hidden wind.

In [ ]:
# First, we put all stocks on an equal playing field so we can compare them fairly
standardized_returns = (returns - returns.mean()) / returns.std()

# Next, we ask the computer to find the top 12 hidden "winds" (Factors)
n_components = 12
pca = PCA(n_components=n_components)
pca.fit(standardized_returns)


Let's look at a chart that shows how important each of these hidden winds is. The first wind usually explains a huge chunk of why stocks move, while the 12th wind might only explain a tiny detail.

In [ ]:
# Calculate how important each hidden wind is
exp_var = pca.explained_variance_ratio_
cum_exp_var = np.cumsum(exp_var)

plt.figure(figsize=(10, 5))
plt.bar(range(1, n_components + 1), exp_var * 100, alpha=0.7, color='steelblue', label='Importance of this single wind (%)')
plt.step(range(1, n_components + 1), cum_exp_var * 100, where='mid', color='red', marker='o', label='Total importance combined (%)')
plt.xlabel("Hidden Wind (Factor Number)")
plt.ylabel("How much stock movement it explains (%)")
plt.title("Which Hidden Winds are the Strongest?")
plt.xticks(range(1, n_components + 1))
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


Now we can look at the table that shows how sensitive the stocks are to these hidden winds.

In [ ]:
# Extract the table showing stock sensitivities to the winds
pca_loadings = pca.components_.T  
loadings_df = pd.DataFrame(pca_loadings, index=tickers, columns=[f"Wind {i+1}" for i in range(n_components)])
print("Sensitivity Table (Showing just the first 5 stocks and first 5 winds):")
print(loadings_df.iloc[:5, :5].round(4))


## Section 3: What is our Portfolio actually betting on?
### 📝 Connecting the Pieces
We built our "perfect mix" of stocks in Section 1. But what is that mix actually doing? Is it secretly placing a massive bet on technology companies? Is it betting on oil prices? 

To find out, we multiply our **Portfolio Recipe** (from Section 1) by the **Sensitivity Table** (from Section 2). 

This gives us a single scorecard. It tells us exactly how exposed our entire portfolio is to each of those hidden winds. If we have a massive positive number for "Wind 1", our portfolio will soar when the overall market goes up, but crash if it goes down.

In [ ]:
# Calculate our portfolio's overall exposure to the hidden winds
portfolio_exposures = np.dot(optimal_weights, pca_loadings)
exposure_series = pd.Series(portfolio_exposures, index=[f"Wind {i+1}" for i in range(n_components)])

print("Our Portfolio's Final Scorecard (Exposure to the 12 Winds):")
print(exposure_series.round(4))


Let's visualize this scorecard. This chart is incredibly valuable because it shows a beginner exactly where their real risks are hidden, even if their portfolio looks "diversified" on the surface.

In [ ]:
# Visualizing the Final Scorecard
plt.figure(figsize=(10, 5))
colors = ['navy' if x > 0 else 'darkred' for x in exposure_series.values]
plt.bar(exposure_series.index, exposure_series.values, color=colors, alpha=0.85, edgecolor='black')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.title("What is our Portfolio Betting On? (Overall Risk Exposures)")
plt.xlabel("Hidden Wind")
plt.ylabel("Amount of Exposure")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
